# rotation-matrix-3d-y-axis — ex9: camera-to-world transform (rays + pose)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d-y-axis`. Running the final beacon cell reports progress against the `Numpy: Applied patterns and advanced` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rotation-matrix-3d-y-axis`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d-y-axis"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Y-axis rotation — quick refresher

**The matrix.** Right-hand rotation by `θ` about Y:
```
R_y(θ) = [[ cos θ,  0,  sin θ],
          [ 0,      1,  0    ],
          [-sin θ,  0,  cos θ]]
```
Anything along Y stays put (middle row `[0,1,0]`); the X-Z plane rotates.

**Acting on data.** Column-vector form: `v' = R @ v`. Batch of row-vectors `(N, 3)`: `points' = points @ R.T`. Composition: `R(α) @ R(β) = R(α + β)` for single-axis rotations; multi-axis rotations don't commute.

**Numerical truth.** Rotation matrices are orthogonal: `R @ R.T = I` and `R.inverse() == R.T`. Floating-point composition accumulates ~1e-7 error per matmul.

### Exercise 9 — camera-to-world transform (rays + pose)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Integrate batched rotation, broadcast addition of a translation, and the camera-vs-world distinction for rays vs points, with step-by-step shape debugging.
> Keywords: camera-pose, world-frame, ray-transform, integrative, shape-debug
> ```

**KCs targeted:** `rotation-matrix-y-construct`, `rotate-batch-of-points`, `rigid-body-transform`

Implement `ex9_cam_to_world(ray_dirs_cam, ray_origins_cam, R_cw, t_cw)` for a NeRF-style ray transform.

**Inputs.**
- `ray_dirs_cam`: `(B, 3)` ray directions in camera frame (unit length, no translation).
- `ray_origins_cam`: `(B, 3)` ray origins in camera frame (a point — gets translated).
- `R_cw`: `(3, 3)` rotation from camera to world.
- `t_cw`: `(3,)` translation of camera origin in world coords.

**Outputs.** A `dict`:
```
{
  'dirs_world':    (B, 3) — directions: rotate only, NO translation
  'origins_world': (B, 3) — points: rotate AND translate (add t_cw)
}
```

**Key insight.** Vectors (directions) only rotate; points (origins) rotate + translate. Mixing this up is the most common bug in any 3-D pipeline.

**Pipeline (build in order, print at each step):**
1. Print `ray_dirs_cam.shape`, `ray_origins_cam.shape`, `R_cw.shape`, `t_cw.shape`.
2. Compute `dirs_world = ray_dirs_cam @ R_cw.T` — print its shape.
3. Compute `origins_world = ray_origins_cam @ R_cw.T + t_cw` (broadcast). Print shape.
4. Return both in the dict.

> ⚠️ **Integrative.** Three concepts: batch rotation, point-vs-vector distinction, broadcast addition. Print shapes between every step — don't trust a one-liner.

In [ ]:
def ex9_cam_to_world(ray_dirs_cam: Tensor, ray_origins_cam: Tensor,
                     R_cw: Tensor, t_cw: Tensor) -> dict:
    """Transform camera-frame rays into world frame. Returns dict with 'dirs_world' and 'origins_world'."""
    raise NotImplementedError()


def _test_ex9():
    import math

    B = 6
    t.manual_seed(11)
    ray_dirs_cam = t.nn.functional.normalize(t.randn(B, 3), dim=-1)
    ray_origins_cam = t.zeros(B, 3)  # camera-frame rays start at the camera origin

    # Camera rotated π/4 about Y, translated to (5, 0, 2) in world coords.
    theta = t.tensor(math.pi / 4)
    c, s = t.cos(theta).item(), t.sin(theta).item()
    R_cw = t.tensor([
        [c,   0.0, s  ],
        [0.0, 1.0, 0.0],
        [-s,  0.0, c  ],
    ])
    t_cw = t.tensor([5.0, 0.0, 2.0])

    out = ex9_cam_to_world(ray_dirs_cam, ray_origins_cam, R_cw, t_cw)

    # Structure.
    assert set(out.keys()) == {'dirs_world', 'origins_world'}, \
        f'expected dirs_world+origins_world, got {sorted(out.keys())}'

    # Shapes.
    assert out['dirs_world'].shape == (B, 3), f'dirs shape {out["dirs_world"].shape}'
    assert out['origins_world'].shape == (B, 3), f'origins shape {out["origins_world"].shape}'

    # Origins (all zero in camera frame) must all map to exactly t_cw in world frame.
    for i in range(B):
        assert t.allclose(out['origins_world'][i], t_cw, atol=1e-6), \
            f'origin {i}: expected {t_cw}, got {out["origins_world"][i]}'

    # Direction lengths must be preserved (rotation is orthogonal).
    norms_cam = ray_dirs_cam.pow(2).sum(dim=-1).sqrt()
    norms_world = out['dirs_world'].pow(2).sum(dim=-1).sqrt()
    assert t.allclose(norms_cam, norms_world, atol=1e-6), \
        'direction lengths should be preserved through rotation'

    # Critical: directions must NOT have t_cw added to them.
    dirs_with_translation = ray_dirs_cam @ R_cw.T + t_cw
    assert not t.allclose(out['dirs_world'], dirs_with_translation, atol=1e-3), \
        'BUG: directions should rotate only — you accidentally added translation!'

    # Hand-check: a camera-frame ray pointing along -Z (canonical "forward")
    # under R_cw(π/4) about Y should be (-sin(π/4), 0, -cos(π/4)) in world frame.
    fwd_cam = t.tensor([[0.0, 0.0, -1.0]])
    fwd_world = ex9_cam_to_world(fwd_cam, t.zeros(1, 3), R_cw, t_cw)['dirs_world'][0]
    expected_fwd = t.tensor([-math.sin(math.pi / 4), 0.0, -math.cos(math.pi / 4)])
    assert t.allclose(fwd_world, expected_fwd, atol=1e-6), \
        f'forward ray rotation mismatch: {fwd_world} vs {expected_fwd}'

    # Step-by-step shape debug printout.
    print('=== shapes pipeline ===')
    print(f'  ray_dirs_cam:    {tuple(ray_dirs_cam.shape)}')
    print(f'  ray_origins_cam: {tuple(ray_origins_cam.shape)}')
    print(f'  R_cw:            {tuple(R_cw.shape)}')
    print(f'  t_cw:            {tuple(t_cw.shape)}')
    print(f'  -> dirs_world:    {tuple(out["dirs_world"].shape)}  (rotated, no translate)')
    print(f'  -> origins_world: {tuple(out["origins_world"].shape)}  (rotated + translated)')
    print()
    print(f'all origins land at t_cw = {t_cw.tolist()}: True')
    print(f'direction norms preserved: {t.allclose(norms_cam, norms_world)}')
    _dd_passed.add('ex9')
    print("ex9 ✓")

_test_ex9()

<details><summary>Solution</summary>

```python
def ex9_cam_to_world(ray_dirs_cam: Tensor, ray_origins_cam: Tensor,
                     R_cw: Tensor, t_cw: Tensor) -> dict:
    # Directions: rotate only.  (B, 3) @ (3, 3).T -> (B, 3)
    dirs_world = ray_dirs_cam @ R_cw.T
    # Origins: rotate, then add the camera translation in world coords.
    # t_cw is (3,); broadcasts over the B axis.
    origins_world = ray_origins_cam @ R_cw.T + t_cw
    return {'dirs_world': dirs_world, 'origins_world': origins_world}
```

**The vector-vs-point distinction.** A direction has no position — it's the difference of two points. Translating it would be a category error: `(p₁ + t) - (p₂ + t) = p₁ - p₂`, the translation cancels. So directions get rotation only. Points (origins) carry position and get the full rigid transform `R @ p + t`.

**Where this lives in real code.** Every NeRF/3DGS/SfM pipeline has this exact function near the top of its ray-generation step. Get it wrong and your scene appears at the right orientation but completely the wrong location — a famously hard-to-debug failure mode because *most* test cases (single translation, single rotation) still pass.

**Homogeneous-coordinates alternative.** You can pack rotation+translation into a `(4, 4)` matrix and represent points as `(x, y, z, 1)` vs directions as `(x, y, z, 0)` — the trailing 0 zeros out the translation column automatically. Same math, less code, but harder to debug shape-wise.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()